## 25차 여행 소비 예측을 위한 학습 데이터셋 구축 실습

### 1. 데이터 셋 불러오기

In [6]:
# 1. 데이터 셋 불러오기
# 구조 확인
# 상위 5개 행 확인

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt


CSV_PATH = "17차시_tn_traveller_master_with_total_top20.csv"
TARGET = "top_spender"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VALID_SIZE = 0.2

df_org = pd.read_csv(CSV_PATH)

print('1단계 - 원본 데이터 크기(shape):', df_org.shape)
df_org.head()

df = df_org.copy()



1단계 - 원본 데이터 크기(shape): (3200, 27)


### 2. 데이터 정제 및 파생 변수 생성

In [3]:
# 2. 데이터 정제 및 파생 변수 생성

# 데이터 정제 및 파생 변수 생성

#  TRIP_DAYS 생성 
# df['TRAVEL_STATUS_YMD']
if "TRAVEL_STATUS_YMD" in df.columns:
    date_parts = df["TRAVEL_STATUS_YMD"].str.split("~", n=1, expand=True)
    start = pd.to_datetime(date_parts[0].str.strip(), errors="coerce")
    end = pd.to_datetime(date_parts[1].str.strip(), errors="coerce")
    df["TRIP_DAYS"] = (end - start).dt.days + 1

print(f'TRIP_DAYS 생성 완료 {df["TRIP_DAYS"].count()}개')

# df["TRIP_DAYS"]

TRIP_DAYS 생성 완료 3200개


In [4]:

# 지역명 통일
df["TRAVEL_STATUS_RESIDENCE"]
region_map = {
    '서울특별시':'서울', '부산광역시':'부산', '대구광역시':'대구',
    '인천광역시':'인천', '광주광역시':'광주', '대전광역시':'대전',
    '울산광역시':'울산', '세종특별자치시':'세종', '경기도':'경기',
    '강원도':'강원', '강원특별자치도':'강원', '충청북도':'충북',
    '충청남도':'충남', '전라북도':'전북', '전북특별자치도':'전북',
    '전라남도':'전남', '경상북도':'경북', '경상남도':'경남',
    '제주특별자치도':'제주'
}
for col in ['TRAVEL_STATUS_RESIDENCE', 'TRAVEL_STATUS_DESTINATION']:
    if col in df.columns:
        # 변경 전 값 저장
        # before = df[col].copy()

        # 지역명 변경
        df[col] = df[col].replace(region_map)

        # 실제로 변경된 값만 출력
        # changed = before != df[col]

        # print(f"\n[{col}] 변경 내역")
        # print(pd.DataFrame({
        #     "변경 전": before[changed],
        #     "변경 후": df.loc[changed, col]
        # }))
# df["TRAVEL_STATUS_RESIDENCE"]
# df["TRAVEL_STATUS_DESTINATION"]

In [5]:
# 드랍할 컬럼
DROP_COLS = [
    "TRAVELER_ID",
    "RESIDENCE_SGG_CD",
    "TRAVEL_LIKE_SIDO_1",
    "TRAVEL_LIKE_SGG_1",
    "TRAVEL_STATUS_YMD",
    "total_spend"
]

drop_cols_exist = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=drop_cols_exist)
print("드랍한 컬럼: ", drop_cols_exist)
print("드랍 후 데이터 크기: ", df.shape)

드랍한 컬럼:  ['TRAVELER_ID', 'RESIDENCE_SGG_CD', 'TRAVEL_LIKE_SIDO_1', 'TRAVEL_LIKE_SGG_1', 'TRAVEL_STATUS_YMD', 'total_spend']
드랍 후 데이터 크기:  (3200, 22)


In [9]:
# 결측치 확인
missing_before = pd.DataFrame({
    '결측치 수': df.isna().sum(),
    '결측 비율(%)': df.isna().mean().mul(100)
}).query('`결측치 수` > 0').sort_values('결측 비율(%)', ascending=False)
missing_before.round(2)

,결측치 수,결측 비율(%)
JOB_ETC,3059,95.59
HOUSE_INCOME,844,26.38
TRAVEL_MOTIVE_3,251,7.84
JOB_NM,141,4.41
TRAVEL_MOTIVE_2,123,3.84
MARR_STTS,2,0.06
EDU_FNSH_SE,1,0.03


In [ ]:
# 범주형 

In [ ]:
#결측치가 대부분인 JOB_ETC 제거
df_clean = df.drop(columns=['JOB_ETC']).copy()

# 범주형은 결측치에 최빈값으로 대체
category_cols = ['TRAVEL_MOTIVE_3', 'JOB_NM', 'TRAVEL_MOTIVE_2', 'MARR_STTS', 'EDU_FNSH_SE']
for col in category_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0])

# 수치형은 평균으로 하면 이상치 때문에 데이터가 신뢰하기 어려워 중앙값으로 대체
df_clean['HOUSE_INCOME'] = df_clean['HOUSE_INCOME'].fillna(df_clean['HOUSE_INCOME'].median())

# print('처리 후 결측치:', df_clean.isna().sum())
print('처리 후 전체 결측치 수:', df_clean.isna().sum().sum())


처리 후 전체 결측치 수: 0


### 3단계. One-Hot Encoding 적용

In [26]:
# 원-핫 인코딩할 컬럼 (범주형 컬럼 숫자의 크기는 의미 없음)
ONE_COLS = [
    "GENDER",
    "JOB_NM",
    "TRAVEL_MOTIVE_1",
    "MARR_STTS",
    "TRAVEL_STATUS_RESIDENCE",
    "TRAVEL_STATUS_DESTINATION",
    "TRAVEL_STATUS_ACCOMPANY"
]
df_encoded = pd.get_dummies(
    df_clean, columns=ONE_COLS, drop_first=True, dtype=int
)

assert not df_encoded.columns.duplicated().any(), '중복 컬럼명이 있습니다.'
missing = df_encoded.isna().sum()
print(missing[missing > 0])
assert int(df_encoded.isna().sum().sum()) == 0, '인코딩 후 결측치가 남아 있습니다.'
print('3단계 - One-Hot Encoding 후 데이터 크기(shape):', df_encoded.shape)

Series([], dtype: int64)
3단계 - One-Hot Encoding 후 데이터 크기(shape): (3200, 80)


### 4. 입력 변수와 타깃 변수 분리

In [36]:
target_col = 'top20_spender'
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col].astype(int)

# 타깃변수 고유값 확인
print('타깃 고유값:', y.unique())

# 타깃변수 결측치 확인
print('타깃 결측치 수:', y.isna().sum())

assert set(y.unique()).issubset({0, 1}), '타깃은 0과 1로 구성되어야 합니다.'
print('입력 변수 X:', X.shape)
print('타깃 y:', y.shape)

타깃 고유값: [0 1]
타깃 결측치 수: 0
입력 변수 X: (3200, 79)
타깃 y: (3200,)


### 5. 데이터셋 분리

In [28]:
# 5단계. Train / Validation / Test 데이터셋 분리
# Test: 전체의 20%, Validation: 나머지 80% 중 20%
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALID_SIZE = 0.2

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=VALID_SIZE, random_state=RANDOM_STATE, stratify=y_temp
)

print('5단계 - Train 데이터 크기:', X_train.shape, y_train.shape)
print('5단계 - Validation 데이터 크기:', X_valid.shape, y_valid.shape)
print('5단계 - Test 데이터 크기:', X_test.shape, y_test.shape)
print('타깃 비율 - Train/Valid/Test:',
      round(y_train.mean(), 4), round(y_valid.mean(), 4), round(y_test.mean(), 4))

5단계 - Train 데이터 크기: (2048, 79) (2048,)
5단계 - Validation 데이터 크기: (512, 79) (512,)
5단계 - Test 데이터 크기: (640, 79) (640,)
타깃 비율 - Train/Valid/Test: 0.1982 0.1992 0.1984


### 6. 수치형 변수 스케일링

In [29]:
# 6단계. 수치형 변수 스케일링
# 원핫 더미 컬럼 제거한 스케일링 필요 컬럼
X_train.columns

# 원-핫 인코딩으로 생성된 컬럼 찾기
dummy_cols = [
    col for col in X_train.columns
    if any(col.startswith(f"{one_col}_") for one_col in ONE_COLS)
]

# 수치형 컬럼
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# 원-핫 컬럼을 제외한 실제 스케일링 대상
scale_cols = [
    col for col in numeric_cols
    if col not in dummy_cols
]

print("전체 컬럼 수:", len(X_train.columns))
print("원-핫 컬럼 수:", len(dummy_cols))
print("스케일링 컬럼 수:", len(scale_cols))

print("\n전체:")
print(X_train.columns)
print("\n스케일링 대상:")
print(scale_cols)


전체 컬럼 수: 79
원-핫 컬럼 수: 61
스케일링 컬럼 수: 16

전체:
Index(['TRAVELER_ID', 'RESIDENCE_SGG_CD', 'AGE_GRP', 'EDU_NM', 'EDU_FNSH_SE',
       'FAMILY_MEMB', 'INCOME', 'HOUSE_INCOME', 'TRAVEL_TERM', 'TRAVEL_NUM',
       'TRAVEL_LIKE_SIDO_1', 'TRAVEL_LIKE_SGG_1', 'TRAVEL_STATUS_YMD',
       'TRAVEL_MOTIVE_2', 'TRAVEL_MOTIVE_3', 'TRAVEL_COMPANIONS_NUM',
       'TRAVEL_STYLE_MODE', 'total_spend', 'GENDER_여', 'JOB_NM_2.0',
       'JOB_NM_3.0', 'JOB_NM_4.0', 'JOB_NM_5.0', 'JOB_NM_6.0', 'JOB_NM_7.0',
       'JOB_NM_8.0', 'JOB_NM_9.0', 'JOB_NM_10.0', 'JOB_NM_11.0', 'JOB_NM_12.0',
       'JOB_NM_13.0', 'TRAVEL_MOTIVE_1_2', 'TRAVEL_MOTIVE_1_3',
       'TRAVEL_MOTIVE_1_4', 'TRAVEL_MOTIVE_1_5', 'TRAVEL_MOTIVE_1_6',
       'TRAVEL_MOTIVE_1_7', 'TRAVEL_MOTIVE_1_8', 'TRAVEL_MOTIVE_1_9',
       'TRAVEL_MOTIVE_1_10', 'MARR_STTS_2.0', 'MARR_STTS_3.0', 'MARR_STTS_4.0',
       'MARR_STTS_5.0', 'TRAVEL_STATUS_RESIDENCE_경기도',
       'TRAVEL_STATUS_RESIDENCE_경상남도', 'TRAVEL_STATUS_RESIDENCE_경상북도',
       'TRAVEL_STATUS_RE

In [30]:
# 스케일링 적용 후 데이터 크기 확인
scaler = StandardScaler()

float_cast = {c: 'float64' for c in scale_cols}
X_train_scaled = X_train.copy().astype(float_cast)
X_valid_scaled = X_valid.copy().astype(float_cast)
X_test_scaled = X_test.copy().astype(float_cast)
X_train_scaled.loc[:, scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_valid_scaled.loc[:, scale_cols] = scaler.transform(X_valid[scale_cols])
X_test_scaled.loc[:, scale_cols] = scaler.transform(X_test[scale_cols])

print('6단계 - 스케일링 컬럼 수:', len(scale_cols))
print('6단계 - 스케일링 적용 후 Train/Valid/Test:',
      X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

X_train_scaled.head()

6단계 - 스케일링 컬럼 수: 16
6단계 - 스케일링 적용 후 Train/Valid/Test: (2048, 79) (512, 79) (640, 79)


,TRAVELER_ID,RESIDENCE_SGG_CD,AGE_GRP,EDU_NM,EDU_FNSH_SE,FAMILY_MEMB,INCOME,HOUSE_INCOME,TRAVEL_TERM,TRAVEL_NUM,...,TRAVEL_STATUS_DESTINATION_전남,TRAVEL_STATUS_DESTINATION_제주,TRAVEL_STATUS_DESTINATION_충남,TRAVEL_STATUS_DESTINATION_충북,TRAVEL_STATUS_ACCOMPANY_2인 여행(가족 외),TRAVEL_STATUS_ACCOMPANY_3대 동반 여행(친척 포함),TRAVEL_STATUS_ACCOMPANY_3인 이상 여행(가족 외),TRAVEL_STATUS_ACCOMPANY_나홀로 여행,TRAVEL_STATUS_ACCOMPANY_부모 동반 여행,TRAVEL_STATUS_ACCOMPANY_자녀 동반 여행
625,a006914,-1.218426,-0.040512,1.472052,-0.369675,-0.530083,0.464004,1.632446,-0.715330,-0.779895,...,0,0,0,0,0,0,0,0,0,0
1982,a003057,0.114853,-1.016623,2.753294,0.756953,-0.530083,-0.039056,-1.490418,0.882266,0.169421,...,0,0,0,0,1,0,0,0,0,0
1185,a006726,0.886751,-1.016623,0.190810,-0.369675,0.262333,0.464004,1.632446,-0.715330,0.169421,...,0,0,0,0,0,0,0,1,0,0
308,a002158,0.956924,-1.016623,-2.371674,-0.369675,0.262333,-0.542115,-1.936542,0.882266,0.169421,...,0,0,0,0,0,0,0,1,0,0
288,a000335,0.886751,1.911709,0.190810,-0.369675,1.847165,-1.548235,2.078569,0.882266,1.118738,...,0,0,0,0,0,0,1,0,0,0


In [31]:
# MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()

X_train_scaled_tmp = X_train.copy().astype(float_cast)
X_valid_scaled_tmp = X_valid.copy().astype(float_cast)
X_test_scaled_tmp = X_test.copy().astype(float_cast)
X_train_scaled_tmp.loc[:, scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_valid_scaled_tmp.loc[:, scale_cols] = scaler.transform(X_valid[scale_cols])
X_test_scaled_tmp.loc[:, scale_cols] = scaler.transform(X_test[scale_cols])

print('6단계 - MinMax 스케일링 컬럼 수:', len(scale_cols))
print('6단계 - MinMax 스케일링 적용 후 Train/Valid/Test:',
      X_train_scaled_tmp.shape, X_valid_scaled_tmp.shape, X_test_scaled_tmp.shape)
X_train_scaled_tmp.head()

6단계 - MinMax 스케일링 컬럼 수: 16
6단계 - MinMax 스케일링 적용 후 Train/Valid/Test: (2048, 79) (512, 79) (640, 79)


,TRAVELER_ID,RESIDENCE_SGG_CD,AGE_GRP,EDU_NM,EDU_FNSH_SE,FAMILY_MEMB,INCOME,HOUSE_INCOME,TRAVEL_TERM,TRAVEL_NUM,...,TRAVEL_STATUS_DESTINATION_전남,TRAVEL_STATUS_DESTINATION_제주,TRAVEL_STATUS_DESTINATION_충남,TRAVEL_STATUS_DESTINATION_충북,TRAVEL_STATUS_ACCOMPANY_2인 여행(가족 외),TRAVEL_STATUS_ACCOMPANY_3대 동반 여행(친척 포함),TRAVEL_STATUS_ACCOMPANY_3인 이상 여행(가족 외),TRAVEL_STATUS_ACCOMPANY_나홀로 여행,TRAVEL_STATUS_ACCOMPANY_부모 동반 여행,TRAVEL_STATUS_ACCOMPANY_자녀 동반 여행
625,a006914,-1.218426,-0.040512,1.472052,-0.369675,-0.530083,0.464004,1.632446,-0.715330,-0.779895,...,0,0,0,0,0,0,0,0,0,0
1982,a003057,0.114853,-1.016623,2.753294,0.756953,-0.530083,-0.039056,-1.490418,0.882266,0.169421,...,0,0,0,0,1,0,0,0,0,0
1185,a006726,0.886751,-1.016623,0.190810,-0.369675,0.262333,0.464004,1.632446,-0.715330,0.169421,...,0,0,0,0,0,0,0,1,0,0
308,a002158,0.956924,-1.016623,-2.371674,-0.369675,0.262333,-0.542115,-1.936542,0.882266,0.169421,...,0,0,0,0,0,0,0,1,0,0
288,a000335,0.886751,1.911709,0.190810,-0.369675,1.847165,-1.548235,2.078569,0.882266,1.118738,...,0,0,0,0,0,0,1,0,0,0


### 7. 최종 데이터셋 저장 및 검증

In [34]:
# 7단계. 최종 데이터셋 저장 및 검증
from pathlib import Path

output_dir = Path('결과물')
output_dir.mkdir(exist_ok=True)

dataset_train = pd.concat([X_train_scaled, y_train], axis=1)
dataset_valid = pd.concat([X_valid_scaled, y_valid], axis=1)
dataset_test = pd.concat([X_test_scaled, y_test], axis=1)

outputs = {
    'dataset_train.csv': dataset_train,
    'dataset_valid.csv': dataset_valid,
    'dataset_test.csv': dataset_test,
}
for filename, dataset in outputs.items():
    # 결과물 폴더에 데이터 셋을 파일로 저장
    path = output_dir / filename
    dataset.to_csv(path, index=False, encoding='utf-8-sig')

    # 파일 읽기
    reloaded = pd.read_csv(path)
    assert reloaded.shape == dataset.shape, f'{filename} 저장 크기가 다릅니다.'
    assert target_col in reloaded.columns, f'{filename}에 타깃이 없습니다.'
    assert int(reloaded.isna().sum().sum()) == 0, f'{filename}에 결측치가 있습니다.'
    print(f'저장·재검증 완료: {path} | shape={reloaded.shape} | 결측치 개수={reloaded.isna().sum().sum()}')

print('7단계 - 모든 데이터셋 저장 완료')

저장·재검증 완료: 결과물\dataset_train.csv | shape=(2048, 80) | 결측치 개수=0
저장·재검증 완료: 결과물\dataset_valid.csv | shape=(512, 80) | 결측치 개수=0
저장·재검증 완료: 결과물\dataset_test.csv | shape=(640, 80) | 결측치 개수=0
7단계 - 모든 데이터셋 저장 완료
